# Round 14 — Term saturation and reference-length normalization

One bounded feature experiment. No neural inference, downloads or ensemble search. The existing development cohort is exploratory, not a fresh holdout or Kaggle score.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/bm25_support_features.json").is_file())
from scripts.run_bm25_support_features import run_study, figures, write_dashboard
config = json.loads((ROOT / "configs/bm25_support_features.json").read_text())
print("Primary:",config["primary"],"| New fits:",config["new_fits"])
print("Candidate feature columns:",config["new_primary_columns"])


Primary: bm25_all | New fits: 12
Candidate feature columns: 48


## Evidence and fixed hypothesis

Term-frequency saturation and reference-length normalization change existing comparisons, not labels or neural weights. The bounded similarity is normalized for query length; it is a BM25-inspired feature, not raw search-engine BM25. The same count vocabularies are reused for TF-IDF and b=0 controls.

No companion-round results are read.

In [2]:
for n,slug in [(12,"reference_consistency_features"),(13,"passage_support_features")]:
    prior=json.loads((ROOT / "reports" / slug / "results.json").read_text())
    print("Round",n,"decision:",prior["decision"])
    display(pd.DataFrame(prior["pooled_metrics"]))


Round 12 decision: DO_NOT_PROMOTE_PRIMARY


,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,context_evidence,0.729257,0.736183,0.741330
4,uniform_all,0.728910,0.735914,0.740913
5,agreement_features,0.727830,0.734316,0.738909
6,reciprocity_features,0.726878,0.732264,0.738835
7,consistency_all,0.726598,0.730519,0.736879
8,quality_null_all,0.723669,0.709782,0.738809
9,geometry_only_all,0.719054,0.724195,0.730477


Round 13 decision: DO_NOT_PROMOTE_PRIMARY


,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,qwen_raw,0.719893,0.739666,0.738959
1,answer_only,0.719893,0.738680,0.738959
2,frozen_basic,0.723068,0.733852,0.737090
3,context_evidence,0.729257,0.736183,0.741330
4,uniform_all,0.728910,0.735914,0.740913
5,word_passages,0.730255,0.738013,0.744573
6,character_passages,0.724681,0.734648,0.738145
7,passage_all,0.724384,0.735518,0.739711
8,whole_comment_all,0.726949,0.739874,0.742511
9,boundary_null_all,0.723435,0.733203,0.737058


## Verified compute or completed-result reuse

Use the terminal helper first. It bounds the scientific process before executing this notebook. This cell only reads its verified result and never starts a new fit.

In [3]:
result=json.loads((ROOT / "reports/bm25_support_features/results.json").read_text())
print("Run:",result["run_id"],"Decision:",result["decision"])
print("Cached control parity:",result["control_design_parity"])
CHARTS=figures(result)


Run: beb42e731d2d910d7ef5 Decision: DO_NOT_PROMOTE_PRIMARY
Cached control parity: True


## Per-policy performance and uncertainty

A higher mean cannot conceal a policy regression. Intervals condition on fixed predictions and do not cover all earlier adaptive choices.

In [4]:
display(pd.DataFrame(result["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")
CHARTS[1].show(renderer="plotly_mimetype")

,fold,policy,variant,auc,brier,log_loss
0,0,"No Advertising: Spam, referral links, unsolici...",qwen_raw,0.679254,0.250238,0.760003
1,1,No legal advice: Do not offer or request legal...,qwen_raw,0.760533,0.221528,0.709879
2,0,"No Advertising: Spam, referral links, unsolici...",answer_only,0.679254,0.260747,0.828707
3,1,No legal advice: Do not offer or request legal...,answer_only,0.760533,0.228357,0.767258
4,0,"No Advertising: Spam, referral links, unsolici...",frozen_basic,0.693097,0.254469,0.827292
5,1,No legal advice: Do not offer or request legal...,frozen_basic,0.753038,0.234327,0.821515
6,0,"No Advertising: Spam, referral links, unsolici...",context_evidence,0.703470,0.247081,0.823188
7,1,No legal advice: Do not offer or request legal...,context_evidence,0.755044,0.235380,0.840367
8,0,"No Advertising: Spam, referral links, unsolici...",uniform_all,0.703246,0.247805,0.821082
9,1,No legal advice: Do not offer or request legal...,uniform_all,0.754574,0.233871,0.817539


## Mechanism controls and family ablations

The primary cannot be swapped for a favorable secondary candidate.

In [5]:
display(pd.DataFrame(result["comparisons"]))
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

,candidate,reference,comparison,delta_auc,simultaneous_low,simultaneous_high
0,word_bm25,context_evidence,word_bm25 vs context anchor,-0.005432,-0.029063,0.018199
1,character_bm25,context_evidence,character_bm25 vs context anchor,-0.012029,-0.035659,0.011602
2,bm25_all,context_evidence,bm25_all vs context anchor,-0.020698,-0.044329,0.002932
3,tfidf_all,context_evidence,tfidf_all vs context anchor,-0.007903,-0.031534,0.015728
4,no_length_all,context_evidence,no_length_all vs context anchor,-0.015931,-0.039562,0.007699
5,label_null_all,context_evidence,label_null_all vs context anchor,-0.013836,-0.037467,0.009795
6,bm25_all,qwen_raw,Primary vs qwen_raw,-0.011335,-0.034965,0.012296
7,bm25_all,frozen_basic,Primary vs frozen_basic,-0.014509,-0.038140,0.009122
8,bm25_all,uniform_all,Primary vs uniform_all,-0.020352,-0.043982,0.003279
9,bm25_all,tfidf_all,Primary vs tfidf_all,-0.012796,-0.036426,0.010835


## Feature coverage and explicit missing evidence

Zero lexical overlap has zero similarity, not an invented label. Inspect coverage before interpreting averages.

In [6]:
display(pd.DataFrame(result["diagnostics"]))
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

,fold,inner_fold,self_overlap,mode,family,reference_rows,vocabulary_columns,mean_reference_length,maximum_reference_length,zero_query_fraction,mean_similarity,query_used_for_vocabulary
0,0,0,0,bm25,word,419,6000,41.682578,161.0,0.0,0.020246,False
1,0,0,0,tfidf,word,419,6000,41.682578,161.0,0.0,0.022657,False
2,0,0,0,no_length,word,419,6000,41.682578,161.0,0.0,0.020124,False
3,0,0,0,label_null,word,419,6000,41.682578,161.0,0.0,0.020246,False
4,0,0,0,bm25,character,419,6000,317.312649,1251.0,0.0,0.031198,False
...,...,...,...,...,...,...,...,...,...,...,...,...
59,1,outer_query,0,label_null,word,366,6000,71.912568,189.0,0.0,0.025569,False
60,1,outer_query,0,bm25,character,366,6000,394.912568,992.0,0.0,0.041212,False
61,1,outer_query,0,tfidf,character,366,6000,394.912568,992.0,0.0,0.077599,False
62,1,outer_query,0,no_length,character,366,6000,394.912568,992.0,0.0,0.043321,False


## Associations and probability diagnostics

Coefficients are fitted associations, not causal explanations. AUC improvement does not guarantee log-loss improvement.

In [7]:
CHARTS[6].show(renderer="plotly_mimetype")
CHARTS[7].show(renderer="plotly_mimetype")

## Fixed decision, provenance and next gate

A pass is eligibility for later independent validation, not model promotion. Feature research remains open.

In [8]:
print("Decision:",result["decision"])
for limitation in result["limitations"]:
    print(limitation)
print("Dashboard:",write_dashboard(ROOT,result))
print("GitHub is not changed by this notebook.")

Decision: DO_NOT_PROMOTE_PRIMARY
This repeatedly inspected development cohort is not independent confirmation.
The unpromoted Round 9 context-evidence anchor was chosen adaptively before these rounds.
New reference statistics are group-cross-fitted; inherited adapted answer training margins remain in-sample.
No semantic counterpart labels or passage labels are invented; original supplied reference labels only.
Simultaneous intervals cover this round, not the entire adaptive project search.
Both Rounds 14 and 15 were specified before running either. Round 15 is not an input.
Term-frequency saturation and reference-length normalization change existing comparisons, not labels or neural weights. The bounded similarity is normalized for query length; it is a BM25-inspired feature, not raw search-engine BM25. The same count vocabularies are reused for TF-IDF and b=0 controls.
Dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/bm25_support_features/dashboard.html
GitHub i